In [1]:
!python --version

Python 3.11.13


재구성

In [1]:
!pip uninstall -y tf-keras keras-nightly keras==3.* tensorflow==2.16.* tensorflow==2.17.* tensorflow==2.18.* tf-nightly


Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
Found existing installation: tensorflow 2.15.0.post1
Uninstalling tensorflow-2.15.0.post1:
  Successfully uninstalled tensorflow-2.15.0.post1


In [2]:
!pip -q install "numpy==1.26.4" "ml-dtypes==0.2.0" "h5py==3.10.0"


In [3]:
%env TF_USE_LEGACY_KERAS=1
!pip -q install "tensorflow==2.15.0.post1" "keras==2.15.0"


env: TF_USE_LEGACY_KERAS=1


In [4]:
!pip -q install --no-deps "deepctr==0.9.3"


In [5]:
import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# 내부 경로에 Keras 2 심볼 매핑
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# init_ops_v2 대체 (DeepCTR가 참조)
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

print("TF:", tf.__version__, "| Keras:", keras.__version__, "→ shim ready")


TF: 2.15.0 | Keras: 2.15.0 → shim ready


In [6]:
import tensorflow as tf, keras, deepctr
print("TF:", tf.__version__)
print("Keras:", keras.__version__)
print("DeepCTR:", deepctr.__version__)

from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
from deepctr.models import DeepFM

print("DeepCTR import OK")


TF: 2.15.0
Keras: 2.15.0
DeepCTR: 0.9.3
DeepCTR import OK


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


In [36]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
df = train.copy()

In [38]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

    max_len : 800
    topk : 1000
    min_count = 500

In [39]:
# padding
# from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences

MAX_LEN = 150

def parse_inc_to_array(s: str) -> np.ndarray:
    # "1,2,3" -> [1,2,3] (int32), 음수 제외 +1 (0은 pad용으로 남김)
    a = np.fromstring(str(s), sep=',', dtype=np.int32)
    if a.size == 0:
        return a
    return a[a >= 0] + 1

# seq → np.ndarray 리스트 (메모리에서만 보관)
seq_arrs = df["seq"].map(parse_inc_to_array)

# padding 배열 (df에 저장하지 않음)
seq_padded = pad_sequences(
    seq_arrs.tolist(),
    maxlen=MAX_LEN,
    dtype='int32',
    padding='post',
    truncating='pre',
    value=0
)

# seq_len만 df에 저장 (필수)
df["seq_len"] = seq_arrs.map(len).clip(upper=MAX_LEN).astype("int32")

### 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.

In [40]:
label_feat = ['gender', 'age_group', 'day_of_week']

encoders = {}

for feat in label_feat:
    le = LabelEncoder()
    df[feat] = le.fit_transform(df[feat])
    encoders[feat] = le

In [41]:
HASH_BUCKET = 1_000_000
MAX_LEN = 150

EMBED_DIM = 8 # 임시로 통일

sparse_fixed = [
    SparseFeat('gender', vocabulary_size= df["gender"].nunique() , embedding_dim= EMBED_DIM),
    SparseFeat('age_group' , vocabulary_size= df["age_group"].nunique() , embedding_dim= EMBED_DIM),
    SparseFeat('day_of_week', vocabulary_size=df["day_of_week"].nunique() , embedding_dim= EMBED_DIM),
    SparseFeat('hour', vocabulary_size=24 , embedding_dim=EMBED_DIM),
]

sparse_hash = [
    SparseFeat('inventory_id' , vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , embedding_name = 'lala' , use_hash= True),
    SparseFeat('l_feat_14', vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , use_hash= True ),
]

ordinal_sparse = [
    SparseFeat('l_feat_3' ,vocabulary_size= 3 , embedding_dim= EMBED_DIM),
    SparseFeat('l_feat_27',vocabulary_size= 5 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_e_4' ,vocabulary_size= 4 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_1' ,vocabulary_size= 5 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_3' ,vocabulary_size= 6 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_4' ,vocabulary_size= 6 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_8' ,vocabulary_size= 7 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_13',vocabulary_size= 5 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_16',vocabulary_size= 7 , embedding_dim= EMBED_DIM),
    SparseFeat('feat_a_18',vocabulary_size= 7 , embedding_dim= EMBED_DIM)
]

# ordinal_scores = [
#     DenseFeat('l_feat_3_ordscore', 1),
#     DenseFeat('feat_a_1_ordscore', 1),


varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size= HASH_BUCKET , #2715931
                            embedding_dim= EMBED_DIM ,
                            use_hash=True ,
                            embedding_name = 'lala'),
    maxlen = MAX_LEN,
    combiner = 'mean',
    length_name= 'seq_len',
    weight_name = None,
    weight_norm = False
)


seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

# dense feats 자동 수집
nominal_names = [f.name for f in (sparse_fixed + sparse_hash)]
ordinal_names = [f.name for f in ordinal_sparse]

exclude = set(nominal_names + ordinal_names + [label_col, seq_col, seq_len_col])
dense_feats = [DenseFeat(c, 1) for c in df.columns if c not in exclude]



In [42]:
# 연속형 인코딩
dense_feat_names = [f.name for f in dense_feats]

mms = MinMaxScaler(feature_range=(0, 1))
df[dense_feat_names] = mms.fit_transform(df[dense_feat_names])


In [43]:
linear_feature_columns = sparse_fixed + sparse_hash + ordinal_sparse + dense_feats
dnn_feature_columns    = linear_feature_columns + [varlen_seq]

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용

# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭


## 학습 샘플 생성 및 모델 학습

DeepCTR 모델은 내부적으로 특성 이름별로 Input Layer를 자동 생성한다.

그래서 입력을 dict 형태로 요구한다.


In [44]:
target = 'clicked'

train, test = train_test_split(df, test_size=0.2, random_state=2020)

train_model_input = {name: train[name] for name in feature_names}
test_model_input = {name: test[name] for name in feature_names}

train_y = train[target].to_numpy()
test_y = test[target].to_numpy()

In [45]:
model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model.compile("adam", "binary_crossentropy",
              metrics=['binary_crossentropy'], )

In [46]:
history = model.fit(train_model_input, train[target].values,
                    batch_size=256, epochs=10, verbose=2, validation_split=0.2, )
pred_ans = model.predict(test_model_input, batch_size=256)


print("test LogLoss", round(log_loss(test[target].values, pred_ans), 4))
print("test AUC", round(roc_auc_score(test[target].values, pred_ans), 4))

AttributeError: module 'tensorflow.python.distribute.input_lib' has no attribute 'DistributedDatasetInterface'